# Задание

1. Реализовать функции активации из презентации
    * Тождественная
    * Еденичная ступенька
    * Логистическая (сигмоида или гладкая ступенька)
    * th
    * arctg

2. Изучить, как изменяется поведение однослойного перцептрона при изменении функции активации

3. Ответить на вопросы в конце ноутбука

# Импорт библиотек

Загружаем необходимые библиотеки для обработки данных, визуализации и машинного обучения.

In [28]:
%pip install pandas numpy plotly nbformat scikit-learn datasets


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [29]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from datasets import load_dataset

# Реализуем функции активации

In [30]:
# Тождественная
def identical(x : float) -> float:
    return x

# Еденичная ступенька
def single_step(x : float) -> float:
    return 1 if x > 0 else 0

# Логистическая функция
def logistic_function(x : float) -> float:
    return 1 / (1 + np.exp(-x))

# th
def th_function(x : float) -> float:
    return np.tanh(x)

# arctg
def arctg_function(x : float) -> float:
    return np.arctan(x)

# Исходные данные

In [31]:
SIZE = 100
SEED = 45

In [32]:
np.random.seed(SEED)

class_1 = np.random.randn(SIZE, 2) + np.array([-2, -2])
class_2 = np.random.randn(SIZE, 2) + np.array([2, 2])

X = np.vstack((class_1, class_2))
y = np.array([0]*SIZE + [1]*SIZE)



In [33]:
fig = go.Figure()


fig.add_trace(go.Scatter(
    x=X[:SIZE, 0], 
    y=X[:SIZE, 1],
    mode='markers',
    marker=dict(size=8, color='blue'),
    name='Class 0'
))

fig.add_trace(go.Scatter(
    x=X[SIZE:, 0],
    y=X[SIZE:, 1],
    mode='markers',
    marker=dict(size=8, color='red'),
    name='Class 1'
))

fig.update_layout(
    title='Распределение данных',
    xaxis_title='X1',
    yaxis_title='X2',
    hovermode='closest',
    width=700,
    height=700
)

fig.show()


# Решение с помощью однослойного перцептрона

In [ ]:
from typing import Callable
import numpy as np

def solve_single_perceptron(
    X, 
    y, 
    activation_function: Callable[[float], float] = logistic_function,
    EPOCHS: int = 10,
    LEARNING_RATE: float = 0.1,
    ) -> tuple[np.ndarray, float]:
    """
    Функция для обучения однослойного перцептрона.

    Args:
        X (np.ndarray): Входные данные размером (n_samples, n_features).
        y (np.ndarray): Целевые значения размером (n_samples,).
        activation_function (Callable[[float], float]): Функция активации.
        EPOCHS (int): Количество эпох обучения.
        LEARNING_RATE (float): Скорость обучения.

    Returns:
        weights (np.ndarray): Обученные веса нейрона.
        bias (float): Обученный смещающий член.
    """

    w = np.random.randn(2)
    b = 0.0

    for epoch in range(EPOCHS):
        for i in range(len(X)):
            z = activation_function(np.dot(X[i], w) + b)
            y_pred = 1 if z >= 0 else 0
            error = y[i] - y_pred
            w += LEARNING_RATE * error * X[i]
            b += LEARNING_RATE * error

        preds = np.array([1 if activation_function(np.dot(x, w) + b) >= 0 else 0 for x in X])
        acc = (preds == y).mean()

        print(f"Эпоха: {epoch + 1}/{EPOCHS}\t | Точность: {acc:.4f}")

    return w , b


def plot_boundart(
    weights: np.ndarray, 
    bias: float, 
    activation_function_name: str
    ) -> None:
    """ 
    Функция для построения границы принятия решений нейрона.

    Args:
        weights (np.ndarray): Веса нейрона.
        bias (float): Смещающий член нейрона.
        activation_function_name (str): Название функции активации.
    
    Returns:
        None
    """
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x_plot = np.array([x_min, x_max])

    w0, w1 = weights[0], weights[1]
    if abs(w1) > 1e-8:
        y_plot = -(w0 * x_plot + bias) / w1
        boundary = go.Scatter(x=x_plot, y=y_plot, mode='lines', line=dict(dash='dash', color='black'), name='Граница решений')
    else:
        # вертикальная граница, если w1 ~ 0
        x_vert = -bias / w0 if abs(w0) > 1e-8 else x_min
        boundary = go.Scatter(x=[x_vert, x_vert], y=[X[:, 1].min() - 1, X[:, 1].max() + 1], mode='lines', line=dict(dash='dash', color='black'), name='Граница решений')

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=X[y == 0, 0],
        y=X[y == 0, 1],
        mode='markers',
        marker=dict(size=8, color='blue'),
        name='Class 0'
    ))
    fig.add_trace(go.Scatter(
        x=X[y == 1, 0],
        y=X[y == 1, 1],
        mode='markers',
        marker=dict(size=8, color='red'),
        name='Class 1'
    ))
    fig.add_trace(boundary)

    fig.update_layout(
        title=f'Граница решений однослойного перцептрона {activation_function_name}',
        xaxis_title='x1',
        yaxis_title='x2',
        hovermode='closest',
        width=700,
        height=700
    )

    fig.show()

## Решение с тождестенной активацией

In [35]:
weights, bias = solve_single_perceptron(
    X, 
    y, 
    activation_function=identical,
)
plot_boundart(weights, bias, 'Тождественная')

Эпоха 1/10	 | точность: 0.9950
Эпоха 2/10	 | точность: 0.9950
Эпоха 3/10	 | точность: 0.9950
Эпоха 4/10	 | точность: 0.9900
Эпоха 5/10	 | точность: 0.9950
Эпоха 6/10	 | точность: 0.9950
Эпоха 7/10	 | точность: 0.9950
Эпоха 8/10	 | точность: 0.9950
Эпоха 9/10	 | точность: 1.0000
Эпоха 10/10	 | точность: 1.0000


## Решение с еденичной ступенькой

In [36]:
weights, bias = solve_single_perceptron(
    X, 
    y, 
    activation_function=single_step,
)
plot_boundart(weights, bias, 'Единчная ступенька')

Эпоха 1/10	 | точность: 0.5000
Эпоха 2/10	 | точность: 0.5000
Эпоха 3/10	 | точность: 0.5000
Эпоха 4/10	 | точность: 0.5000
Эпоха 5/10	 | точность: 0.5000
Эпоха 6/10	 | точность: 0.5000
Эпоха 7/10	 | точность: 0.5000
Эпоха 8/10	 | точность: 0.5000
Эпоха 9/10	 | точность: 0.5000
Эпоха 10/10	 | точность: 0.5000


## Решение с сигмоидой

In [37]:
weights, bias = solve_single_perceptron(
    X,
    y,
    activation_function=logistic_function,
)
plot_boundart(weights, bias, 'Логистическая функция')

Эпоха 1/10	 | точность: 0.5000
Эпоха 2/10	 | точность: 0.5000
Эпоха 3/10	 | точность: 0.5000
Эпоха 4/10	 | точность: 0.5000
Эпоха 5/10	 | точность: 0.5000
Эпоха 6/10	 | точность: 0.5000
Эпоха 7/10	 | точность: 0.5000
Эпоха 8/10	 | точность: 0.5000
Эпоха 9/10	 | точность: 0.5000
Эпоха 10/10	 | точность: 0.5000


/var/folders/y7/3khx9vbx74z9_0bd7nswgf080000gn/T/ipykernel_53081/1459870230.py:11: RuntimeWarning:

overflow encountered in exp



## Решение с th

In [38]:
weights, bias = solve_single_perceptron(
    X,
    y,
    activation_function=th_function,
)
plot_boundart(weights, bias, 'Гиперболический тангенс')

Эпоха 1/10	 | точность: 0.9950
Эпоха 2/10	 | точность: 0.9950
Эпоха 3/10	 | точность: 0.9900
Эпоха 4/10	 | точность: 0.9950
Эпоха 5/10	 | точность: 0.9900
Эпоха 6/10	 | точность: 0.9950
Эпоха 7/10	 | точность: 0.9950
Эпоха 8/10	 | точность: 0.9950
Эпоха 9/10	 | точность: 1.0000
Эпоха 10/10	 | точность: 1.0000
Эпоха 3/10	 | точность: 0.9900
Эпоха 4/10	 | точность: 0.9950
Эпоха 5/10	 | точность: 0.9900
Эпоха 6/10	 | точность: 0.9950
Эпоха 7/10	 | точность: 0.9950
Эпоха 8/10	 | точность: 0.9950
Эпоха 9/10	 | точность: 1.0000
Эпоха 10/10	 | точность: 1.0000


## Решение с тождестенной arctg

In [39]:
weights, bias = solve_single_perceptron(
    X,
    y,
    activation_function=arctg_function,
)
plot_boundart(weights, bias, 'Арктангенс')

Эпоха 1/10	 | точность: 0.9950
Эпоха 2/10	 | точность: 0.9950
Эпоха 3/10	 | точность: 0.9950
Эпоха 4/10	 | точность: 0.9850
Эпоха 5/10	 | точность: 0.9950
Эпоха 6/10	 | точность: 0.9950
Эпоха 7/10	 | точность: 0.9950
Эпоха 8/10	 | точность: 0.9950
Эпоха 9/10	 | точность: 1.0000
Эпоха 10/10	 | точность: 1.0000


# Вопросы

## 1. Что делает функция активации в искусственном нейроне

Функция активации решает, какой сигнал пойдет на выход нейрона. Она берет сумму всех входов, умноженных на веса, и преобразует это число. Самое главное — она добавляет нелинейность. Без неё нейросеть была бы просто набором линейных уравнений и не смогла бы решать сложные задачи.


## 2. Чем отличается искусственный нейрон от однослойного перцептрона?

Нейрон — это просто математическая модель (входы * веса -> сумма -> функция активации). А перцептрон — это уже простейшая нейросеть, которая состоит из таких нейронов (обычно одного слоя). По сути, перцептрон — это нейрон + алгоритм его обучения.


## 3. Какое правило используется для обучения перцептрона?

Используется дельта-правило (или правило Розенблатта). Идея простая: мы корректируем веса пропорционально ошибке. Если нейрон ошибся, мы меняем веса так, чтобы в следующий раз ответ был ближе к правильному. Формула: `новый_вес = старый_вес + скорость_обучения * ошибка * вход`.


## 4. Почему однослойный перцептрон не может решить задачу XOR?

Потому что однослойный перцептрон умеет разделять данные только одной прямой линией (линейно). А в задаче XOR точки (0,0) и (1,1) лежат по диагонали от (0,1) и (1,0), и их никак не разделить одной линией. Чтобы решить XOR, нужно добавить скрытый слой.


## 5. Что произойдет, если изменить функцию активации с сигмоиды на ReLU?

Скорее всего, сеть станет обучаться быстрее, потому что ReLU проще считать (это просто `max(0, x)`), и у неё нет проблемы затухающего градиента, как у сигмоиды. Но есть риск получить "мертвые нейроны", которые всегда выдают ноль и перестают обучаться, если веса уйдут сильно в минус.
